# 04 – Regularisation Experiment (weight decay + label smoothing)

**ENGE707 Assessment 3 – Facial Feature Classification Challenge (Kaggle)**

**Purpose.** Test whether two standard regularisation techniques reduce the overfitting seen in stage-2 fine-tuning
(report Section 4.5): **AdamW with weight decay 0.05** and **label smoothing 0.1**. Stage 2 is rerun from the same stage-1
checkpoints for both backbones (EfficientNetB0 and ResNet50V2), each with two seeds (42 and 7), giving four regularised
models. They are compared with the baseline models and combined into ensembles (v4a: 4 regularised models; v4b: 6 models).

**Inputs.** `../models/stage1_best.keras` and `../models/r50_stage1_best.keras` (stage-1 checkpoints from `03_model`),
`../backup_run1` and `../backup_run2` (baseline predictions from the first run and the rerun), `../src/pipeline.py`.

**Outputs.** New models and probability files in `../models/` (`reg_*`), `../submissions/submission_v4_ensemble6.csv`,
`../submissions/submission_v4_reg_ensemble4.csv` (only written if its validation rule is met) and extra rows in
`../docs/experiment_log.md`.

**How to run.** Start Jupyter from the `notebooks/` folder, run `03_model.ipynb` and `03_model_rerun.ipynb` first, then run these
cells top to bottom. Training uses the GPU and takes roughly 8–12 minutes per model.

**Note on the loss values.** With label smoothing 0.1 and eight classes the cross-entropy cannot fall below about 0.47, so
the loss values in this notebook are not comparable with the baseline losses; models are compared by validation errors.

## Step 1: Set up regularised EfficientNetB0 (seed 42)

Loads the stage-1 EfficientNetB0 checkpoint, builds the datasets with one-hot labels (needed for label smoothing), unfreezes the same layers as the baseline (block6, block7, top; BatchNorm stays frozen) and compiles with AdamW (learning rate 1e-4, weight decay 0.05) and categorical cross-entropy with label smoothing 0.1.

In [1]:
import sys; sys.path.append("../src")  # make ../src importable
import numpy as np, tensorflow as tf  # numpy: arrays; TensorFlow: deep learning
from tensorflow.keras import layers  # Keras layers (used to recognise BatchNorm layers)
import pipeline  # shared data module

tf.keras.utils.set_random_seed(42)  # fixed seed (Python, NumPy, TensorFlow) for reproducibility

# same pipeline as before, with one-hot labels so label smoothing can be applied
to_onehot = lambda x, y: (x, tf.one_hot(y, 8))  # turn the integer label into a vector of 8 values (1 for the true class)
train_ds = pipeline.make_dataset(pipeline.train_df, shuffle=True, augment_data=True).map(to_onehot)  # training data: shuffled, augmented, one-hot
val_ds = pipeline.make_dataset(pipeline.val_df, shuffle=False, augment_data=False).map(to_onehot)  # validation data: fixed order, one-hot

model = tf.keras.models.load_model("../models/stage1_best.keras")   # EfficientNetB0 after stage 1
base = model.get_layer("efficientnetb0")  # the backbone inside the model
base.trainable = True  # make the backbone trainable first...
for layer in base.layers:  # ...then choose layer by layer
    layer.trainable = (layer.name.startswith(("block6", "block7", "top"))  # train only the last blocks...
                       and not isinstance(layer, layers.BatchNormalization))  # ...and never the BatchNorm layers

model.compile(  # configure training
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.05),  # AdamW: Adam with decoupled weight decay 0.05
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),  # cross-entropy with targets softened by 0.1
    metrics=["accuracy"],  # track accuracy
)
print("Trainable params:", sum(int(np.prod(w.shape)) for w in model.trainable_weights))  # expect 3,140,132

I0000 00:00:1791785975.983487    1875 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791785976.023778    1875 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791785977.183704    1875 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791785978.754508    1875 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6994 MB mem

Trainable params: 3140132


## Step 2: Train regularised EfficientNetB0 (seed 42, up to 12 epochs)

Fine-tunes for up to 12 epochs with the same callbacks as the baseline (best-model checkpoint, early stopping on validation loss, learning-rate reduction) and saves the final model. Validation accuracy reaches about 0.9953.

In [2]:
import os  # os: create folders
os.makedirs("../models", exist_ok=True)  # make the models folder if needed

callbacks = [  # same callbacks as the baseline stage 2
    tf.keras.callbacks.ModelCheckpoint("../models/reg_stage2_best.keras",  # save the best model to this file
                                       monitor="val_accuracy", save_best_only=True, verbose=1),  # judged by validation accuracy
    tf.keras.callbacks.CSVLogger("../models/reg_stage2_log.csv"),  # log every epoch to CSV
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,  # stop after 3 epochs without a better validation loss...
                                     restore_best_weights=True, verbose=1),  # ...and restore the best weights
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,  # multiply the learning rate by 0.3...
                                         patience=1, min_lr=1e-6, verbose=1),  # ...after 1 stalled epoch, down to 1e-6
]

history = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=callbacks)  # train for at most 12 epochs
model.save("../models/reg_final.keras")   # the weights restored by early stopping

Epoch 1/12


/home/suemon/enge707-kaggle/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1791786062.564674    2001 service.cc:153] XLA service 0x7698600850f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791786062.564702    2001 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3080, Compute Capability 8.6 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1791786063.066561    2001 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791786064.442688    2001 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1791786064.693163    2001 dot_merger.cc:481] Merg

1591/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9569 - loss: 0.6265

I0000 00:00:1791786114.582682    2001 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_21191__.254
E0000 00:00:1791786116.070037    2001 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1791786118.623557    2001 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1592/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9570 - loss: 0.6265

E0000 00:00:1791786137.471585    1999 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.



Epoch 1: val_accuracy improved from None to 0.98115, saving model to ../models/reg_stage2_best.keras

Epoch 1: finished saving model to ../models/reg_stage2_best.keras
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 86s 39ms/step - accuracy: 0.9570 - loss: 0.6265 - val_accuracy: 0.9811 - val_loss: 0.5342 - learning_rate: 1.0000e-04
Epoch 2/12
1589/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9816 - loss: 0.5413
Epoch 2: val_accuracy improved from 0.98115 to 0.98932, saving model to ../models/reg_stage2_best.keras

Epoch 2: finished saving model to ../models/reg_stage2_best.keras
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.9816 - loss: 0.5412 - val_accuracy: 0.9893 - val_loss: 0.5061 - learning_rate: 1.0000e-04
Epoch 3/12
1590/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9872 - loss: 0.5175
Epoch 3: val_accuracy improved from 0.98932 to 0.99175, saving model to ../models/reg_stage2_best.keras

Epoch 3: finished saving model to ../models/reg_stage2_best.keras
1592/1592 ━━━

## Step 3: Evaluate regularised EfficientNetB0 and compare with the baselines

Saves the validation and test probabilities of this model, then prints its error count and compares it with the four baseline single models and with a few ensembles that include it.

In [3]:
y = pipeline.val_df["label"].values  # true validation classes
pv = model.predict(val_ds, verbose=0)          # labels in val_ds are ignored by predict
pt = model.predict(pipeline.make_test_dataset(), verbose=1)  # probabilities for the test images
np.save("../models/reg_val_probs.npy", pv); np.save("../models/reg_test_probs.npy", pt)  # save both for the ensembles

load = lambda run, f: np.load(f"../{run}/models/{f}")  # helper: load a saved array from a backup folder
errs = lambda q: int((q.argmax(axis=1) != y).sum())  # helper: number of validation errors of a probability table
p = {"run1_eff": load("backup_run1", "val_probs.npy"), "run1_r50": load("backup_run1", "val_probs_r50.npy"),  # baseline run-1 models
     "run2_eff": load("backup_run2", "val_probs.npy"), "run2_r50": load("backup_run2", "val_probs_r50.npy")}  # baseline rerun models

print(f"Regularised EfficientNetB0 errors: {errs(pv)} | accuracy {1 - errs(pv) / len(y):.4f}")  # this model's errors and accuracy
print("Baseline single models:", {k: errs(v) for k, v in p.items()})  # errors of the four baseline models
print("4-model ensemble:", errs(sum(p.values()) / 4))  # baseline four-model average
print("5-model ensemble (4 + regularised):", errs((sum(p.values()) + pv) / 5))  # baseline four plus this model
print("Regularised EfficientNetB0 + run-1 ResNet50V2:", errs((pv + p["run1_r50"]) / 2))  # two-model combination

921/921 ━━━━━━━━━━━━━━━━━━━━ 23s 22ms/step
Regularised EfficientNetB0 errors: 60 | accuracy 0.9953
Baseline single models: {'run1_eff': 73, 'run1_r50': 63, 'run2_eff': 75, 'run2_r50': 75}
4-model ensemble: 43
5-model ensemble (4 + regularised): 44
Regularised EfficientNetB0 + run-1 ResNet50V2: 40


## Step 4: Regularised EfficientNetB0, second seed (seed 7)

Repeats the previous three steps with seed 7 (set-up, training, saving predictions) to measure seed-to-seed variation, then compares the two regularised runs (60 and 60 errors) and their average (57).

In [4]:
tf.keras.utils.set_random_seed(7)  # different seed from the first run
model2 = tf.keras.models.load_model("../models/stage1_best.keras")  # start again from the same stage-1 checkpoint
base2 = model2.get_layer("efficientnetb0"); base2.trainable = True  # backbone, made trainable
for layer in base2.layers:  # choose trainable layers one by one
    layer.trainable = (layer.name.startswith(("block6", "block7", "top"))  # last blocks only...
                       and not isinstance(layer, layers.BatchNormalization))  # ...never BatchNorm
model2.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.05),  # AdamW, weight decay 0.05
               loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])  # label smoothing 0.1

cb2 = [tf.keras.callbacks.ModelCheckpoint("../models/reg2_stage2_best.keras", monitor="val_accuracy",  # save best model to this file
                                          save_best_only=True, verbose=0),  # keep only the best; stay quiet
       tf.keras.callbacks.CSVLogger("../models/reg2_stage2_log.csv"),  # log every epoch
       tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),  # early stopping
       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-6, verbose=1)]  # LR reduction
model2.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb2)  # train for at most 12 epochs
model2.save("../models/reg2_final.keras")  # save the final (best-restored) model

pv2 = model2.predict(val_ds, verbose=0)  # validation probabilities of the second run
np.save("../models/reg2_val_probs.npy", pv2)  # save them
np.save("../models/reg2_test_probs.npy", model2.predict(pipeline.make_test_dataset(), verbose=0))  # predict and save test probabilities
print("Regularised run 2 errors:", errs(pv2), "| run 1:", errs(pv))  # compare the two seeds
print("Both regularised runs averaged:", errs((pv + pv2) / 2))  # error count of their average

Epoch 1/12


I0000 00:00:1791787400.281126    1999 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_178489__.254


1590/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9584 - loss: 0.6243

I0000 00:00:1791787442.705991    1998 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_178489__.254


1592/1592 ━━━━━━━━━━━━━━━━━━━━ 69s 33ms/step - accuracy: 0.9584 - loss: 0.6242 - val_accuracy: 0.9865 - val_loss: 0.5236 - learning_rate: 1.0000e-04
Epoch 2/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - accuracy: 0.9829 - loss: 0.5390 - val_accuracy: 0.9870 - val_loss: 0.5057 - learning_rate: 1.0000e-04
Epoch 3/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - accuracy: 0.9875 - loss: 0.5170 - val_accuracy: 0.9895 - val_loss: 0.5003 - learning_rate: 1.0000e-04
Epoch 4/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - accuracy: 0.9903 - loss: 0.5053 - val_accuracy: 0.9921 - val_loss: 0.4919 - learning_rate: 1.0000e-04
Epoch 5/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.9925 - loss: 0.4979 - val_accuracy: 0.9918 - val_loss: 0.4904 - learning_rate: 1.0000e-04
Epoch 6/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.9932 - loss: 0.4937 - val_accuracy: 0.9912 - val_loss: 0.4897 - learning_rate: 1.0000e-04
Epoch 7/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/

## Step 5: Six-model ensemble and Kaggle submission v4b

Averages six models with equal weights (the four baseline models plus the two regularised EfficientNetB0 models) and writes `submission_v4_ensemble6.csv`. This ensemble had 45 validation errors, above the pre-set rule of 'submit only if below 43', but was submitted anyway as an exploratory submission (Kaggle public score 0.9800). Format checks and the number of predictions changed compared with v3 are printed.

In [5]:
import pandas as pd  # pandas: build and read the submission tables
y = pipeline.val_df["label"].values  # true validation classes
errs = lambda q: int((q.argmax(axis=1) != y).sum())  # helper: validation errors of a probability table
lv = lambda run, f: np.load(f"../{run}/models/{f}")  # helper: load a saved array from a backup folder

v = [lv("backup_run1", "val_probs.npy"), lv("backup_run1", "val_probs_r50.npy"),  # validation probabilities: baseline run 1 (EfficientNet, ResNet)
     lv("backup_run2", "val_probs.npy"), lv("backup_run2", "val_probs_r50.npy"), pv, pv2]  # baseline rerun, then the 2 regularised EfficientNets
t = [lv("backup_run1", "test_probs.npy"), lv("backup_run1", "test_probs_r50.npy"),  # test probabilities, same six models in the same order
     lv("backup_run2", "test_probs.npy"), lv("backup_run2", "test_probs_r50.npy"),
     np.load("../models/reg_test_probs.npy"), np.load("../models/reg2_test_probs.npy")]

print("Baseline EfficientNetB0 pair (run 1 + rerun):", errs((v[0] + v[2]) / 2))  # baseline EfficientNetB0 pair
print("Regularised EfficientNetB0 pair:", errs((v[4] + v[5]) / 2))  # regularised EfficientNetB0 pair
e6 = sum(v) / 6  # equal-weight average of the six validation probability tables
print(f"6-model equal-weight ensemble: {errs(e6)} errors | accuracy {1 - errs(e6) / len(y):.4f}")  # validation result of the 6-model ensemble

t6 = sum(t) / 6  # equal-weight average of the six test probability tables
sub = pd.DataFrame({"image_id": pipeline.test_df["image_id"].values,  # test ids in Kaggle's order
                    "class": [pipeline.idx_to_class[i] for i in t6.argmax(axis=1)]})  # predicted class names
sub.to_csv("../submissions/submission_v4_ensemble6.csv", index=False)  # write the submission file
sample = pd.read_csv("../data/sample_submission.csv"); v3 = pd.read_csv("../submissions/submission_v3_ensemble4.csv")  # Kaggle template and v3 for comparison
print("Shape:", sub.shape, "| columns match:", list(sub.columns) == list(sample.columns),  # shape and column check
      "| IDs match:", (sub["image_id"].values == sample["image_id"].values).all())  # id order check
print(f"Predictions changed vs v3: {(sub['class'].values != v3['class'].values).sum()} of {len(sub)}")  # how many predictions differ from v3

Baseline EfficientNetB0 pair (run 1 + rerun): 66
Regularised EfficientNetB0 pair: 57
6-model equal-weight ensemble: 45 errors | accuracy 0.9965
Shape: (29443, 2) | columns match: True | IDs match: True
Predictions changed vs v3: 106 of 29443


## Step 6: Add rows 11–14 to the experiment log

Inserts the four regularisation rows below the existing table in `../docs/experiment_log.md` and appends a note. **Run once only** (a second run duplicates the rows). The row-14 text written here says 'not submitted' because the 6-model ensemble was submitted afterwards as v4b (public score 0.9800): update that row by hand in the log file. Rows 15–18 (the regularised ResNet50V2 runs and the four-model ensemble) are not written by this notebook and are also added by hand.

In [6]:
path = "../docs/experiment_log.md"  # the experiment log file
lines = open(path).read().split("\n")  # read the file as a list of lines
last_row = max(k for k, l in enumerate(lines) if l.startswith("| "))  # index of the last table row
rows = [  # the four new table rows (rows 11-14)
 "| 11 | EfficientNetB0 stage 2 with AdamW (weight decay 0.05) + label smoothing 0.1, seed 42 (from the stage-1 checkpoint of #1) | 0.9953 | - | 60 | not submitted |",
 "| 12 | Same as #11, seed 7 | 0.9953 | - | 60 | not submitted |",
 "| 13 | Average of #11 and #12 (baseline pair #2 + #7: 66 errors) | 0.9955 | - | 57 | not submitted |",
 "| 14 | Equal-weight average of 6 models (#2, #4, #7, #8, #11, #12) | 0.9965 | - | 45 | not submitted (rule fixed in advance: submit only if below 43) |",
]
for i, r in enumerate(rows): lines.insert(last_row + 1 + i, r)  # insert each row just below the table, keeping their order
lines.append("Rows 11-14: regularisation experiment. Regularised runs repeated at 60 errors versus 73 and 75 for the baselines; the 6-model ensemble did not beat the 43-error 4-model ensemble.")  # explanatory note
open(path, "w").write("\n".join(lines))  # write the updated log back to the file
print(open(path).read())  # print the file as a check

# Experiment log (ENGE707 Kaggle)

| # | Model / setup | Val acc | Val loss | Val errors | Kaggle public |
|---|---|---|---|---|---|
| 1 | EfficientNetB0 stage 1 (frozen, lr 1e-3, 5 epochs) | 0.9457 | 0.1517 | - | - |
| 2 | EfficientNetB0 stage 2 (top blocks unfrozen, lr 1e-4, ReduceLROnPlateau, 12 epochs) | 0.9943 | 0.0206 | 73 | 0.9767 (v1) |
| 3 | ResNet50V2 stage 1 (frozen, lr 1e-3, 5 epochs) | 0.9687 | 0.0966 | - | - |
| 4 | ResNet50V2 stage 2 (conv5 unfrozen, lr 1e-4, early stopped epoch 11, restored epoch 8) | 0.9951 | 0.0189 | 63 | not submitted |
| 5 | EfficientNetB0 + horizontal-flip TTA | 0.9943 | - | 73 | not submitted |
| 6 | Ensemble: average of #2 and #4 probabilities | 0.9964 | 0.0138 (log loss) | 46 | 0.9786 (v2) |
| 7 | Rerun of #2 (EfficientNetB0, identical code, seed 42 set) | 0.9941 | - | 75 | not submitted |
| 8 | Rerun of #4 (ResNet50V2, identical code, seed 42 set) | 0.9941 | - | 75 | not submitted |
| 9 | Rerun of #6 (ensemble of #7 and #8) | 0.9958 | - | 53 | 

## Step 7: Set up regularised ResNet50V2 (seed 42)

Same idea as the EfficientNetB0 set-up, for ResNet50V2: load the stage-1 checkpoint, unfreeze the `conv5` stage (BatchNorm stays frozen) and compile with AdamW (weight decay 0.05) and label smoothing 0.1. Expect 14,966,792 trainable parameters.

In [7]:
tf.keras.utils.set_random_seed(42)  # fixed seed
model_r = tf.keras.models.load_model("../models/r50_stage1_best.keras")   # ResNet50V2 after stage 1
base_r = model_r.get_layer("resnet50v2"); base_r.trainable = True  # the backbone, made trainable
for layer in base_r.layers:  # choose trainable layers one by one
    layer.trainable = (layer.name.startswith("conv5")  # train the last residual stage (conv5)...
                       and not isinstance(layer, layers.BatchNormalization))  # ...but never BatchNorm layers
model_r.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.05),  # AdamW with weight decay 0.05
                loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])  # label smoothing 0.1
print("Trainable params:", sum(int(np.prod(w.shape)) for w in model_r.trainable_weights))  # expect 14,966,792

Trainable params: 14966792


## Step 8: Train regularised ResNet50V2 (seed 42) and save its predictions

Fine-tunes for up to 12 epochs with the usual callbacks, saves the model and its validation/test probabilities and prints the validation errors (64) next to the baseline ResNet50V2 runs (63 and 75).

In [8]:
cb = [tf.keras.callbacks.ModelCheckpoint("../models/reg_r50_stage2_best.keras", monitor="val_accuracy",  # save the best model to this file
                                         save_best_only=True, verbose=0),  # keep only the best; stay quiet
      tf.keras.callbacks.CSVLogger("../models/reg_r50_stage2_log.csv"),  # log every epoch
      tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),  # early stopping
      tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-6, verbose=1)]  # LR reduction

model_r.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb)  # train for at most 12 epochs
model_r.save("../models/reg_r50_final.keras")  # save the final model

errs = lambda q: int((q.argmax(axis=1) != pipeline.val_df["label"].values).sum())  # helper: validation errors
pvr = model_r.predict(val_ds, verbose=0)  # validation probabilities
np.save("../models/reg_r50_val_probs.npy", pvr)  # save them
np.save("../models/reg_r50_test_probs.npy", model_r.predict(pipeline.make_test_dataset(), verbose=0))  # predict and save test probabilities
print("Regularised ResNet50V2 run 1 errors:", errs(pvr), "| baseline ResNet50V2 runs: 63 and 75")  # compare with the baseline runs

Epoch 1/12


/home/suemon/enge707-kaggle/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1791789697.176668    2002 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_325957__.169


1591/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9754 - loss: 0.5541

I0000 00:00:1791789752.600549    1998 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_325957__.169


1592/1592 ━━━━━━━━━━━━━━━━━━━━ 90s 50ms/step - accuracy: 0.9754 - loss: 0.5541 - val_accuracy: 0.9880 - val_loss: 0.5001 - learning_rate: 1.0000e-04
Epoch 2/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 62s 39ms/step - accuracy: 0.9877 - loss: 0.5064 - val_accuracy: 0.9887 - val_loss: 0.4981 - learning_rate: 1.0000e-04
Epoch 3/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 62s 39ms/step - accuracy: 0.9914 - loss: 0.4947 - val_accuracy: 0.9917 - val_loss: 0.4901 - learning_rate: 1.0000e-04
Epoch 4/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 59s 37ms/step - accuracy: 0.9932 - loss: 0.4901 - val_accuracy: 0.9937 - val_loss: 0.4854 - learning_rate: 1.0000e-04
Epoch 5/12
1591/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9943 - loss: 0.4869
Epoch 5: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 61s 38ms/step - accuracy: 0.9943 - loss: 0.4869 - val_accuracy: 0.9902 - val_loss: 0.4906 - learning_rate: 1.0000e-04
Epoch 6/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 63s 39ms/step 

## Step 9: Regularised ResNet50V2, second seed (seed 7)

Repeats the ResNet50V2 set-up and training with seed 7 and saves its predictions. Validation errors: 59 (versus 64 for seed 42).

In [9]:
tf.keras.utils.set_random_seed(7)  # different seed from the first ResNet run
model_r2 = tf.keras.models.load_model("../models/r50_stage1_best.keras")  # same stage-1 checkpoint as before
base_r2 = model_r2.get_layer("resnet50v2"); base_r2.trainable = True  # backbone, made trainable
for layer in base_r2.layers:  # choose trainable layers one by one
    layer.trainable = (layer.name.startswith("conv5")  # conv5 stage only...
                       and not isinstance(layer, layers.BatchNormalization))  # ...never BatchNorm
model_r2.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=0.05),  # AdamW, weight decay 0.05
                 loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])  # label smoothing 0.1

cb2 = [tf.keras.callbacks.ModelCheckpoint("../models/reg_r50b_stage2_best.keras", monitor="val_accuracy",  # save best model to this file
                                          save_best_only=True, verbose=0),  # keep only the best; stay quiet
       tf.keras.callbacks.CSVLogger("../models/reg_r50b_stage2_log.csv"),  # log every epoch
       tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),  # early stopping
       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=1, min_lr=1e-6, verbose=1)]  # LR reduction
model_r2.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=cb2)  # train for at most 12 epochs
model_r2.save("../models/reg_r50b_final.keras")  # save the final model

pvr2 = model_r2.predict(val_ds, verbose=0)  # validation probabilities
np.save("../models/reg_r50b_val_probs.npy", pvr2)  # save them
np.save("../models/reg_r50b_test_probs.npy", model_r2.predict(pipeline.make_test_dataset(), verbose=0))  # predict and save test probabilities
print("Regularised ResNet50V2 run 2 errors:", errs(pvr2), "| run 1:", errs(pvr))  # compare the two seeds

Epoch 1/12


I0000 00:00:1791790686.390835    2001 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_465565__.169


1591/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9749 - loss: 0.5548

I0000 00:00:1791790739.381868    2000 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_465565__.169


1592/1592 ━━━━━━━━━━━━━━━━━━━━ 75s 43ms/step - accuracy: 0.9749 - loss: 0.5548 - val_accuracy: 0.9892 - val_loss: 0.5017 - learning_rate: 1.0000e-04
Epoch 2/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 62s 39ms/step - accuracy: 0.9879 - loss: 0.5057 - val_accuracy: 0.9892 - val_loss: 0.4947 - learning_rate: 1.0000e-04
Epoch 3/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 59s 37ms/step - accuracy: 0.9913 - loss: 0.4959 - val_accuracy: 0.9915 - val_loss: 0.4889 - learning_rate: 1.0000e-04
Epoch 4/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 61s 38ms/step - accuracy: 0.9928 - loss: 0.4912 - val_accuracy: 0.9922 - val_loss: 0.4873 - learning_rate: 1.0000e-04
Epoch 5/12
1590/1592 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9946 - loss: 0.4860
Epoch 5: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 60s 37ms/step - accuracy: 0.9946 - loss: 0.4860 - val_accuracy: 0.9808 - val_loss: 0.5133 - learning_rate: 1.0000e-04
Epoch 6/12
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 62s 39ms/step 

## Step 10: Four regularised models: validation rule and Kaggle submission v4a

Averages the four regularised models (2 EfficientNetB0 + 2 ResNet50V2). A rule fixed before this step says the ensemble is submitted only if it has at most 38 validation errors (the baseline four-model ensemble had 43). The rule is met (35 errors, accuracy 0.9973), so `submission_v4_reg_ensemble4.csv` is written and format-checked (Kaggle public score 0.9784).

In [10]:
y = pipeline.val_df["label"].values  # true validation classes
errs = lambda q: int((q.argmax(axis=1) != y).sum())  # helper: validation errors

base_r50_pair = (lv("backup_run1", "val_probs_r50.npy") + lv("backup_run2", "val_probs_r50.npy")) / 2  # baseline ResNet50V2 pair (run 1 + rerun)
print("Regularised ResNet50V2 pair:", errs((pvr + pvr2) / 2), "| baseline ResNet50V2 pair:", errs(base_r50_pair))  # pair versus pair

e4r = (pv + pv2 + pvr + pvr2) / 4          # 2 regularised EfficientNetB0 + 2 regularised ResNet50V2
n = errs(e4r)  # validation errors of the four-model regularised ensemble
print(f"Fully regularised 4-model ensemble: {n} errors | accuracy {1 - n / len(y):.4f} | baseline 4-model ensemble: 43")  # result versus baseline
print("Rule (submit only if <= 38 errors):", "MET" if n <= 38 else "NOT MET")  # report whether the pre-set rule is satisfied

if n <= 38:  # only create the submission file when the rule is met
    t4r = sum(np.load(f"../models/{f}") for f in ["reg_test_probs.npy", "reg2_test_probs.npy",  # sum the four regularised models' test probabilities
                                                    "reg_r50_test_probs.npy", "reg_r50b_test_probs.npy"]) / 4  # ...and divide by 4 for the average
    sub = pd.DataFrame({"image_id": pipeline.test_df["image_id"].values,  # test ids in Kaggle's order
                        "class": [pipeline.idx_to_class[i] for i in t4r.argmax(axis=1)]})  # predicted class names
    sub.to_csv("../submissions/submission_v4_reg_ensemble4.csv", index=False)  # write the submission file
    sample = pd.read_csv("../data/sample_submission.csv")  # Kaggle template for the format check
    print("Saved. Shape:", sub.shape, "| columns match:", list(sub.columns) == list(sample.columns),  # shape and column check
          "| IDs match:", (sub["image_id"].values == sample["image_id"].values).all())  # id order check

Regularised ResNet50V2 pair: 55 | baseline ResNet50V2 pair: 59
Fully regularised 4-model ensemble: 35 errors | accuracy 0.9973 | baseline 4-model ensemble: 43
Rule (submit only if <= 38 errors): MET
Saved. Shape: (29443, 2) | columns match: True | IDs match: True
